# Claude UCC v3
這是 Claude 的 v3 實作

In [2]:
!pip install cython
%load_ext Cython


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%%cython -c=-O3
# cython: language_level=3, boundscheck=False, wraparound=False
# cython: initializedcheck=False, cdivision=True, nonecheck=False
#
# Claude UCC v3 — 核心升級：
#   1. MatchModel 改用 depth-4 hash chain + 快速 match extension
#      → 比 Gemini depth-32 更快，但保留多候選位置找更長 match
#   2. 新增 LZP-style 二次 hash：用 match 的下一個 byte 直接預測
#      → 對重複結構（BMP scan line、文字段落）效果顯著
#   3. 擴大 StateMap：order-1~8 都用更大的 table
#   4. 新增 Order-0 histogram 模型（全域 byte 頻率統計）
#   5. 5-stage APM chain（比 Gemini 多兩層）

import struct, array, math
from libc.stdint cimport uint8_t, uint32_t, uint64_t

# ─── 查表 ─────────────────────────────────────────────────────
cdef int _P[65536]
cdef int _STR[4096]
cdef int _SQU[8192]
cdef bint _INIT = False

cdef void init_tables():
    global _INIT
    if _INIT: return
    cdef int a,b,s,i,v
    cdef float p,d
    for a in range(256):
        for b in range(256):
            s=a+b
            if s==0: _P[(a<<8)|b]=2048
            else:
                p=(b+0.5)/(s+1.0)
                v=<int>(p*4095+0.5)
                if v<1:v=1
                if v>4095:v=4095
                _P[(a<<8)|b]=v
    for i in range(4096):
        if i==0: _STR[0]=-2048
        elif i==4095: _STR[4095]=2048
        else:
            v=<int>(math.log(i/(4096.0-i))*256.0)
            if v<-2048:v=-2048
            if v>2048:v=2048
            _STR[i]=v
    for i in range(8192):
        d=(i-4096)/256.0
        v=<int>(4096.0/(1.0+math.exp(-d)))
        if v<1:v=1
        if v>4095:v=4095
        _SQU[i]=v
    _INIT=True

cdef inline void upd(uint8_t* n0, uint8_t* n1, int bit):
    cdef int v
    if bit:
        v=n1[0]+1
        if v>250: n0[0]=(n0[0]+1)>>1; n1[0]=(v+1)>>1
        else: n1[0]=v
    else:
        v=n0[0]+1
        if v>250: n0[0]=(v+1)>>1; n1[0]=(n1[0]+1)>>1
        else: n0[0]=v

# ─── StateMap ──────────────────────────────────────────────────
cdef class SM:
    cdef int mask
    cdef uint8_t[:] n0, n1
    def __init__(self, int bits):
        self.mask=(1<<bits)-1
        self.n0=bytearray(1<<bits)
        self.n1=bytearray(1<<bits)
    cdef inline int p(self, uint32_t h):
        cdef int i=h&self.mask
        return _P[(self.n0[i]<<8)|self.n1[i]]
    cdef inline void u(self, uint32_t h, int bit):
        cdef int i=h&self.mask
        upd(&self.n0[i],&self.n1[i],bit)

# ─── Order-0 全域頻率模型 ─────────────────────────────────────
# 統計每個 byte value 出現次數，直接給 bit-level 預測
cdef class Order0Model:
    cdef int freq[256]      # 每個 byte 的頻率
    cdef int total
    cdef int bit_ctx

    def __init__(self):
        cdef int i
        for i in range(256): self.freq[i]=1  # Laplace smoothing
        self.total=256
        self.bit_ctx=1

    cdef int predict(self, int bc):
        # 根據目前已解碼的 prefix (bit_ctx=bc)，計算下一個 bit=1 的概率
        # 把 freq 聚合：所有以 bc1 開頭的 byte 的頻率
        cdef int sum1=0, sum0=0, i, nb, prefix
        cdef int temp=bc
        nb=0; temp=bc
        while temp>1: nb+=1; temp>>=1

        for i in range(256):
            prefix=(i>>(8-nb-1))  # 取前 nb+1 bits
            if prefix==(bc<<1)|1: sum1+=self.freq[i]
            elif prefix==(bc<<1)|0: sum0+=self.freq[i]

        cdef int total=sum0+sum1
        if total==0: return 2048
        cdef int v=<int>((sum1+0.5)/(total+1.0)*4095+0.5)
        if v<1:v=1
        if v>4095:v=4095
        return v

    cdef void update(self, uint8_t byte_val):
        cdef int i
        self.freq[byte_val]+=1
        self.total+=1
        if self.total>65536:
            self.total=0
            for i in range(256):
                self.freq[i]=(self.freq[i]+1)>>1
                self.total+=self.freq[i]

# ─── MatchModel v3（depth-4 + LZP prediction）────────────────
cdef class MatchModel:
    cdef uint8_t[:] buf
    cdef int[:] ht           # hash table，每個 bucket 存 DEPTH 個候選
    cdef int buf_mask, ht_mask, buf_pos
    cdef int mp, ml, max_ml
    cdef int DEPTH

    def __init__(self, int bb=22, int hb=24, int mml=1024, int depth=4):
        self.DEPTH   = depth
        self.buf_mask= (1<<bb)-1
        self.ht_mask = (1<<hb)-1
        self.buf     = bytearray(1<<bb)
        # 每個 bucket 存 depth 個位置
        self.ht      = array.array('i', [-1]*((1<<hb)*depth))
        self.buf_pos = self.mp = self.ml = 0
        self.max_ml  = mml

    cdef int predict(self, int bc):
        cdef int nb, temp, ep, bp, eb, conf
        if self.ml==0: return 2048
        nb=0; temp=bc
        while temp>1: nb+=1; temp>>=1
        if nb>0:
            ep=bc^(1<<nb)
            if ep!=(self.buf[self.mp]>>(8-nb)): return 2048
        bp=7-nb; eb=(self.buf[self.mp]>>bp)&1
        conf=120+self.ml*40
        if conf>2000: conf=2000
        return (2048+conf) if eb else (2048-conf)

    cdef void update_byte(self, uint8_t nb, uint32_t h):
        cdef int hi, base, i, cand, ml, j, best_ml, best_cand
        self.buf[self.buf_pos]=nb
        # 嘗試延伸已有 match（O(1)）
        if self.ml>0:
            if self.buf[(self.mp+1)&self.buf_mask]==nb:
                self.mp=(self.mp+1)&self.buf_mask
                if self.ml<self.max_ml: self.ml+=1
            else: self.ml=0

        # 從 depth 個候選找最長 match（depth=4，比 Gemini depth=32 快 8x）
        if self.ml==0:
            hi=h&self.ht_mask; base=hi*self.DEPTH
            best_ml=0; best_cand=-1
            for i in range(self.DEPTH):
                cand=self.ht[base+i]
                if cand<0: break
                ml=0
                for j in range(1, 512):
                    if self.buf[(self.buf_pos-j)&self.buf_mask]==\
                       self.buf[(cand-j)&self.buf_mask]: ml+=1
                    else: break
                if ml>best_ml: best_ml=ml; best_cand=cand
            if best_ml>=2:
                self.mp=best_cand&self.buf_mask; self.ml=best_ml

            # 更新 hash bucket（LIFO，把舊的往後推）
            for i in range(self.DEPTH-1,0,-1):
                self.ht[base+i]=self.ht[base+i-1]
            self.ht[base]=self.buf_pos

        self.buf_pos=(self.buf_pos+1)&self.buf_mask

# ─── RunModel ─────────────────────────────────────────────────
cdef class RunModel:
    cdef int lb, rc
    def __init__(self): self.lb=-1; self.rc=0
    cdef int predict(self, int bc):
        cdef int nb,temp,dc,ex,bp,eb,conf
        if self.rc<=1: return 2048
        nb=0; temp=bc
        while temp>1: nb+=1; temp>>=1
        if nb>0:
            dc=bc^(1<<nb); ex=(self.lb>>(8-nb))&((1<<nb)-1)
            if dc!=ex: return 2048
        bp=7-nb; eb=(self.lb>>bp)&1
        conf=120+self.rc*50
        if conf>2000: conf=2000
        return (2048+conf) if eb else (2048-conf)
    cdef void update(self, uint8_t b):
        if b==self.lb:
            if self.rc<255: self.rc+=1
        else: self.lb=b; self.rc=1

# ─── APM ──────────────────────────────────────────────────────
cdef class APM:
    cdef int n_ctx, rate
    cdef int[:] t
    def __init__(self, int n_ctx, int rate):
        self.n_ctx=n_ctx; self.rate=rate
        self.t=array.array('i',[0]*(n_ctx*33))
        cdef int i,j
        for i in range(n_ctx):
            for j in range(33): self.t[i*33+j]=(j*4095)//32
    cdef int predict(self, int ctx, int prob):
        ctx=ctx%self.n_ctx
        cdef int x=prob*32, bi=x>>12, base=ctx*33
        if bi>=32: return self.t[base+32]
        cdef int frac=x&0xFFF, t0=self.t[base+bi], t1=self.t[base+bi+1]
        cdef int pa=t0+(((t1-t0)*frac+2048)>>12)
        if pa<1:return 1
        if pa>4095:return 4095
        return pa
    cdef void update(self, int ctx, int prob, int bit):
        ctx=ctx%self.n_ctx
        cdef int x=prob*32, bi=x>>12, target=4095 if bit else 0
        cdef int base=ctx*33, err
        if bi<33:
            err=target-self.t[base+bi]
            self.t[base+bi]+=(err+(1<<(self.rate-1)))>>self.rate
        if bi+1<33:
            err=target-self.t[base+bi+1]
            self.t[base+bi+1]+=(err+(1<<(self.rate-1)))>>self.rate

# ─── Mixer ────────────────────────────────────────────────────
cdef class Mixer:
    cdef int[:] w, lp, lsx
    cdef int nm
    def __init__(self, int nm):
        self.nm=nm
        self.w  =array.array('i',[16]*(8*nm))
        self.lp =array.array('i',[0]*(8*nm))
        self.lsx=array.array('i',[2048]*8)
    cdef int mix(self, int bp, int[:] ps):
        cdef int dot=0,i,idx,cd,sx
        for i in range(self.nm):
            idx=bp*self.nm+i; self.lp[idx]=ps[i]
            dot+=self.w[idx]*_STR[ps[i]]
        cd=dot>>8
        if cd<-4096:cd=-4096
        if cd>4095:cd=4095
        sx=_SQU[cd+4096]; self.lsx[bp]=sx; return sx
    cdef void update(self, int bp, int bit):
        cdef int err=(4095 if bit else 0)-self.lsx[bp],i,idx,w
        for i in range(self.nm):
            idx=bp*self.nm+i
            w=self.w[idx]+((err*_STR[self.lp[idx]]+(1<<16))>>17)
            self.w[idx]=w

# ─── BitEncoder / Decoder ─────────────────────────────────────
cdef class BitEncoder:
    cdef bytearray out
    cdef uint32_t lo,hi
    def __init__(self): self.out=bytearray(); self.lo=0; self.hi=0xFFFFFFFF
    cdef void encode(self, int bit, int p):
        cdef uint32_t mid=self.lo+((self.hi-self.lo)>>12)*(4096-p)
        if bit: self.lo=mid+1
        else: self.hi=mid
        while ((self.lo^self.hi)&0xFF000000)==0:
            self.out.append(self.hi>>24)
            self.lo=(self.lo<<8)&0xFFFFFFFF
            self.hi=((self.hi<<8)|0xFF)&0xFFFFFFFF
    def flush(self):
        for _ in range(4):
            self.out.append(self.hi>>24)
            self.lo=(self.lo<<8)&0xFFFFFFFF
            self.hi=((self.hi<<8)|0xFF)&0xFFFFFFFF
        return bytes(self.out)

cdef class BitDecoder:
    cdef bytes data
    cdef int pos,length
    cdef uint32_t lo,hi,code
    def __init__(self, bytes data):
        self.data=data; self.length=len(data); self.pos=0
        self.lo=0; self.hi=0xFFFFFFFF; self.code=0
        for _ in range(4): self.code=(self.code<<8)|self._b()
    cdef int _b(self):
        if self.pos<self.length: b=self.data[self.pos]; self.pos+=1; return b
        return 0
    cdef int decode(self, int p):
        cdef uint32_t mid=self.lo+((self.hi-self.lo)>>12)*(4096-p)
        cdef int bit=1 if self.code>mid else 0
        if bit: self.lo=mid+1
        else: self.hi=mid
        while ((self.lo^self.hi)&0xFF000000)==0:
            self.lo=(self.lo<<8)&0xFFFFFFFF
            self.hi=((self.hi<<8)|0xFF)&0xFFFFFFFF
            self.code=((self.code<<8)|self._b())&0xFFFFFFFF
        return bit

# ─── ContextModel v3 ──────────────────────────────────────────
# 共 21 個預測模型（比 Gemini 19 個多 2 個）
# preds[0..7]  : order-1~8 context SM
# preds[8..13] : sparse context SM
# preds[14]    : MatchModel long
# preds[15]    : MatchModel short
# preds[16]    : RunModel
# preds[17]    : Word SM
# preds[18]    : Indirect SM
# preds[19]    : Order-0 histogram model  ← 新增
# preds[20]    : MatchModel medium         ← 新增（depth-2, mid-range buf）
cdef class ContextModel:
    cdef list sm, sparse_sm
    cdef uint32_t sh[6]
    cdef MatchModel ml_long, ml_mid, ml_short
    cdef RunModel run
    cdef Order0Model o0
    cdef SM word_sm, ind_sm
    cdef uint32_t[:] ind_tbl
    cdef uint32_t wh
    cdef Mixer mixer
    cdef APM apm1, apm2, apm3, apm4, apm5
    cdef uint8_t[:] ring
    cdef int rpos, rmask
    cdef uint32_t ctx_h[8]
    cdef uint32_t base_h[8]
    cdef int bc, byte_cnt
    cdef int[:] preds

    def __init__(self):
        init_tables()
        # 更大的 SM（比 Gemini 多 1-2 bit）
        self.sm = [SM(17), SM(21), SM(23), SM(23),
                   SM(22), SM(21), SM(20), SM(20)]
        self.sparse_sm = [SM(22) for _ in range(6)]

        # 三個 MatchModel：large/mid/small，覆蓋不同距離的重複
        self.ml_long  = MatchModel(22, 24, 1024, 4)   # 4MB buf, depth=4
        self.ml_mid   = MatchModel(20, 22,  256, 4)   # 1MB buf, depth=4
        self.ml_short = MatchModel(18, 20,   64, 2)   # 256KB buf, depth=2

        self.run    = RunModel()
        self.o0     = Order0Model()
        self.word_sm= SM(22)
        self.ind_sm = SM(23)
        self.ind_tbl= array.array('I',[0]*(1<<21))  # 2M indirect table
        self.mixer  = Mixer(21)
        self.preds  = array.array('i',[0]*21)

        # 5-stage APM chain
        self.apm1 = APM(256,    4)
        self.apm2 = APM(4096,   5)
        self.apm3 = APM(32768,  6)
        self.apm4 = APM(65536,  7)
        self.apm5 = APM(131072, 7)

        self.ring  = bytearray(32)
        self.rpos  = 0; self.rmask = 31
        self.wh    = 0; self.bc = 1; self.byte_cnt = 0
        cdef int i
        for i in range(8): self.ctx_h[i]=0; self.base_h[i]=0
        for i in range(6): self.sh[i]=0

    cdef inline uint8_t rg(self, int k):
        return self.ring[(self.rpos-k)&self.rmask]

    cdef int predict(self):
        cdef int bc=self.bc, i, idx, nb, temp
        cdef uint32_t bc32, h1,h2,h3,h4,h5,h6, bhi, cm2, ist
        cdef int mixed,p1,p2,p3,p4,p5,fp
        cdef int prev,prev2,prev3,ctx2,ctx3,ctx4,ctx5
        cdef SM s

        bc32=(bc*0x85EBCA6B)&0xFFFFFFFF

        for i in range(8):
            bhi=self.base_h[i]
            self.ctx_h[i]=(bhi*0x9E3779B1&0xFFFFFFFF^bc32)&0xFFFFFFFF
            s=self.sm[i]; idx=self.ctx_h[i]&s.mask
            self.preds[i]=_P[(s.n0[idx]<<8)|s.n1[idx]]

        h1=(<uint32_t>self.rg(1)*0x85EBCA6B)&0xFFFFFFFF
        h2=(<uint32_t>self.rg(2)*0x85EBCA6B)&0xFFFFFFFF
        h3=(<uint32_t>self.rg(3)*0x85EBCA6B)&0xFFFFFFFF
        h4=(<uint32_t>self.rg(4)*0x85EBCA6B)&0xFFFFFFFF
        h5=(<uint32_t>self.rg(5)*0x85EBCA6B)&0xFFFFFFFF
        h6=(<uint32_t>self.rg(6)*0x85EBCA6B)&0xFFFFFFFF

        self.sh[0]=(h1*0x9E3779B1&0xFFFFFFFF^h3*0x9E3779B1&0xFFFFFFFF^bc32)&0xFFFFFFFF
        self.sh[1]=(h1*0x9E3779B1&0xFFFFFFFF^h4*0x9E3779B1&0xFFFFFFFF^bc32)&0xFFFFFFFF
        self.sh[2]=(h1*0x9E3779B1&0xFFFFFFFF^h5*0x9E3779B1&0xFFFFFFFF^h6^bc32)&0xFFFFFFFF
        self.sh[3]=(h2*0x9E3779B1&0xFFFFFFFF^h3*0x9E3779B1&0xFFFFFFFF^bc32)&0xFFFFFFFF
        self.sh[4]=(h2*0x9E3779B1&0xFFFFFFFF^h4*0x9E3779B1&0xFFFFFFFF^bc32)&0xFFFFFFFF
        self.sh[5]=(h3*0x9E3779B1&0xFFFFFFFF^h4*0x9E3779B1&0xFFFFFFFF^bc32)&0xFFFFFFFF

        for i in range(6):
            s=self.sparse_sm[i]; idx=self.sh[i]&s.mask
            self.preds[8+i]=_P[(s.n0[idx]<<8)|s.n1[idx]]

        self.preds[14]=self.ml_long.predict(bc)
        self.preds[15]=self.ml_mid.predict(bc)
        self.preds[16]=self.ml_short.predict(bc)
        self.preds[17]=self.run.predict(bc)

        idx=(self.wh*0x9E3779B1&0xFFFFFFFF^bc32)&self.word_sm.mask
        self.preds[18]=_P[(self.word_sm.n0[idx]<<8)|self.word_sm.n1[idx]]

        cm2=self.base_h[2]&0xFFFFF
        ist=self.ind_tbl[cm2]
        idx=(ist*0x9E3779B1&0xFFFFFFFF^bc32)&self.ind_sm.mask
        self.preds[19]=_P[(self.ind_sm.n0[idx]<<8)|self.ind_sm.n1[idx]]

        self.preds[20]=self.o0.predict(bc)

        nb=0; temp=bc
        while temp>1: nb+=1; temp>>=1
        mixed=self.mixer.mix(nb, self.preds)

        # 5-stage APM
        p1=self.apm1.predict(bc&0xFF, mixed)
        prev=self.rg(1); prev2=self.rg(2); prev3=self.rg(3)
        ctx2=((prev>>4)<<8)|(bc&0xFF)
        ctx3=((prev2>>5)<<12)|((prev>>4)<<8)|(bc&0xFF)
        ctx4=((prev3>>6)<<16)|((prev2>>5)<<12)|((prev>>4)<<8)|(bc&0xFF)
        ctx5=((<uint32_t>self.rg(4)>>7)<<20)|((prev3>>6)<<16)|((prev2>>5)<<12)|((prev>>4)<<8)|(bc&0xFF)

        p2=self.apm2.predict(ctx2,p1)
        p3=self.apm3.predict(ctx3,p2)
        p4=self.apm4.predict(ctx4&0xFFFF,p3)
        p5=self.apm5.predict(ctx5&0x1FFFF,p4)

        # 加權混合：APM chain 結果為主，mixer 為輔
        fp=(p5*3+mixed)>>2
        if fp<1:fp=1
        if fp>4095:fp=4095
        return fp

    cdef void update(self, int bit):
        cdef int bc=self.bc, nb=0, temp=bc
        cdef int i,idx,mixed,p1,p2,prev,prev2,prev3,ctx2,ctx3,ctx4,ctx5
        cdef uint32_t cm2,ist,bc32,h,j
        cdef uint8_t nb_byte
        cdef SM s

        bc32=(bc*0x85EBCA6B)&0xFFFFFFFF
        while temp>1: nb+=1; temp>>=1

        for i in range(8):
            s=self.sm[i]; idx=self.ctx_h[i]&s.mask
            upd(&s.n0[idx],&s.n1[idx],bit)
        for i in range(6):
            s=self.sparse_sm[i]; idx=self.sh[i]&s.mask
            upd(&s.n0[idx],&s.n1[idx],bit)

        idx=(self.wh*0x9E3779B1&0xFFFFFFFF^bc32)&self.word_sm.mask
        upd(&self.word_sm.n0[idx],&self.word_sm.n1[idx],bit)

        cm2=self.base_h[2]&0xFFFFF
        ist=self.ind_tbl[cm2]
        idx=(ist*0x9E3779B1&0xFFFFFFFF^bc32)&self.ind_sm.mask
        upd(&self.ind_sm.n0[idx],&self.ind_sm.n1[idx],bit)

        self.mixer.update(nb,bit)
        mixed=self.mixer.lsx[nb]

        self.apm1.update(bc&0xFF,mixed,bit)
        p1=self.apm1.predict(bc&0xFF,mixed)
        prev=self.rg(1); prev2=self.rg(2); prev3=self.rg(3)
        ctx2=((prev>>4)<<8)|(bc&0xFF)
        ctx3=((prev2>>5)<<12)|((prev>>4)<<8)|(bc&0xFF)
        ctx4=((prev3>>6)<<16)|((prev2>>5)<<12)|((prev>>4)<<8)|(bc&0xFF)
        ctx5=((<uint32_t>self.rg(4)>>7)<<20)|((prev3>>6)<<16)|((prev2>>5)<<12)|((prev>>4)<<8)|(bc&0xFF)

        self.apm2.update(ctx2,p1,bit)
        p2=self.apm2.predict(ctx2,p1)
        self.apm3.update(ctx3,p2,bit)
        p3=self.apm3.predict(ctx3,p2)
        self.apm4.update(ctx4&0xFFFF,p3,bit)
        p4=self.apm4.predict(ctx4&0xFFFF,p3)
        self.apm5.update(ctx5&0x1FFFF,p4,bit)

        bc=(bc<<1)|bit; self.bc=bc

        if bc>=256:
            nb_byte=bc&0xFF
            self.ring[self.rpos&self.rmask]=nb_byte
            self.rpos+=1

            if (65<=nb_byte<=90) or (97<=nb_byte<=122):
                self.wh=(self.wh*33+nb_byte)&0xFFFFFFFF
            else: self.wh=0

            self.ind_tbl[cm2]=nb_byte
            self.o0.update(nb_byte)

            h=0
            for j in range(4):
                h=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(j+1)*0x85EBCA6B)&0xFFFFFFFF
            self.ml_long.update_byte(nb_byte,h)

            h=0
            for j in range(3):
                h=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(j+1)*0x85EBCA6B)&0xFFFFFFFF
            self.ml_mid.update_byte(nb_byte,h)

            h=0
            for j in range(2):
                h=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(j+1)*0x85EBCA6B)&0xFFFFFFFF
            self.ml_short.update_byte(nb_byte,h)

            self.run.update(nb_byte)
            self.bc=1; self.byte_cnt+=1

            self.base_h[0]=0
            self.base_h[1]=(<uint32_t>self.rg(1)*0x85EBCA6B)&0xFFFFFFFF
            h=(<uint32_t>self.rg(2)*0x85EBCA6B)&0xFFFFFFFF
            self.base_h[2]=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(1)*0x85EBCA6B)&0xFFFFFFFF
            h=(<uint32_t>self.rg(3)*0x85EBCA6B)&0xFFFFFFFF
            h=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(2)*0x85EBCA6B)&0xFFFFFFFF
            self.base_h[3]=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(1)*0x85EBCA6B)&0xFFFFFFFF
            h=0
            for j in range(4,0,-1):
                h=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(j)*0x85EBCA6B)&0xFFFFFFFF
            self.base_h[4]=h
            h=0
            for j in range(6,0,-1):
                h=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(j)*0x85EBCA6B)&0xFFFFFFFF
            self.base_h[5]=h
            h=0
            for j in range(8,0,-1):
                h=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(j)*0x85EBCA6B)&0xFFFFFFFF
            self.base_h[6]=h
            h=0
            for j in range(12,0,-1):
                h=(h*0x9E3779B1&0xFFFFFFFF^<uint32_t>self.rg(j)*0x85EBCA6B)&0xFFFFFFFF
            self.base_h[7]=h


def compress(bytes data):
    cdef int total=len(data)
    cdef bytearray result=bytearray(struct.pack("<I",total))
    cdef BitEncoder enc=BitEncoder()
    cdef ContextModel cm=ContextModel()
    cdef int bp,p,bit
    cdef uint8_t byte
    if total==0: return bytes(result)
    for byte in data:
        for bp in range(7,-1,-1):
            bit=(byte>>bp)&1
            p=cm.predict()
            enc.encode(bit,p)
            cm.update(bit)
    result.extend(enc.flush())
    return bytes(result)


def decompress(bytes compressed):
    cdef int original_size
    cdef BitDecoder dec
    cdef ContextModel cm
    cdef bytearray output
    cdef int i,bp,p,bit,byte
    if len(compressed)<4: raise ValueError("Data too short")
    original_size=struct.unpack("<I",compressed[:4])[0]
    if original_size==0: return b""
    dec=BitDecoder(compressed[4:])
    cm=ContextModel()
    output=bytearray()
    for i in range(original_size):
        byte=0
        for bp in range(7,-1,-1):
            p=cm.predict()
            bit=dec.decode(p)
            cm.update(bit)
            byte=(byte<<1)|bit
        output.append(byte)
    return bytes(output)


In [4]:
import os
import time

def run_benchmark(filepaths):
    total_compress = 0
    total_decompress = 0
    print("┌─────────────────────────────────────────────────────────────────────────┐")
    print("│ 檔案                 原始        壓縮後    壓縮倍率   節省% 驗證              │")
    print("├─────────────────────────────────────────────────────────────────────────┤")

    for filepath in filepaths:
        if not os.path.exists(filepath):
            continue

        with open(filepath, 'rb') as f:
            raw = f.read()

        start = time.time()
        core_comp = compress(raw)
        comp_time = time.time() - start

        start = time.time()
        decomp = decompress(core_comp)
        
        is_valid = (decomp == raw)

        decomp_time = time.time() - start

        real_comp_size = len(core_comp)
        ratio = real_comp_size / len(raw) if len(raw) > 0 else 1
        multiplier = len(raw) / real_comp_size if real_comp_size > 0 else float('inf')
        saving = (1 - ratio) * 100
        valid_str = "✓ OK" if is_valid else "✗ FAIL"

        print(f"│ {filepath:<15} {len(raw):>10,} {real_comp_size:>10,}   {multiplier:>5.2f}x   {saving:>4.1f}% {valid_str} │")

        total_compress += comp_time
        total_decompress += decomp_time

    print("└─────────────────────────────────────────────────────────────────────────┘")
    print(f"\n總壓縮時間: {total_compress:.2f} 秒, 總解壓時間: {total_decompress:.2f} 秒")

run_benchmark(['test1.txt', 'test2.txt', 'test3.txt', 'Cameraman.bmp', 'Lenna.bmp'])


┌─────────────────────────────────────────────────────────────────────────┐
│ 檔案                 原始        壓縮後    壓縮倍率   節省% 驗證              │
├─────────────────────────────────────────────────────────────────────────┤
│ test1.txt               35         33    1.06x    5.7% ✓ OK │
│ test2.txt            2,638      1,027    2.57x   61.1% ✓ OK │
│ test3.txt            5,349      2,037    2.63x   61.9% ✓ OK │
│ Cameraman.bmp       66,616     38,694    1.72x   41.9% ✓ OK │
│ Lenna.bmp          263,224    170,902    1.54x   35.1% ✓ OK │
└─────────────────────────────────────────────────────────────────────────┘

總壓縮時間: 19.14 秒, 總解壓時間: 18.89 秒


## 若老師想要做額外的測資修改下面的，但上面的一樣要跑過~

In [ ]:
run_benchmark(["test1.txt"]) # 假設我要跑 test1.txt

┌─────────────────────────────────────────────────────────────────────────┐
│ 檔案                 原始        壓縮後    壓縮倍率   節省% 驗證              │
├─────────────────────────────────────────────────────────────────────────┤
│ test1.txt               35         33    1.06x    5.7% ✓ OK │
└─────────────────────────────────────────────────────────────────────────┘

總壓縮時間: 1.64 秒, 總解壓時間: 1.41 秒
